# Register Agent to Agentspace

@wangdave

### Prerequisites

- You have an Agentspace application deployed
- You have Agent Engines deployed
- Create a .env file (see below)
- You create OAuth 2.0 client with proper scopes and redirect URL:
https://vertexaisearch.cloud.google.com/oauth-redirect

Example below is based on [Google's OAuth 2.0 client](https://support.google.com/googleapi/answer/6158849):
  - From the GCP console, search for "OAuth consent screen"
  - Create an OAuth 2.0 client for Web application
  - Add the redirect URL specified above
  - Click **Create**
  - Download the `client_secret.json` and upload it here

For other 3rd party OAuth clients, follow their documentation to obtain authorization information.


### 0. Create .env if you don't have one
Fill in the following values in the `.env` file:

In [1]:
from dotenv import load_dotenv
import IPython
import google_auth_oauthlib.flow
import json
import os
import vertexai

### Install required libraries

In [ ]:
!pip3 install -qq -U google-adk "google-cloud-aiplatform[agent_engines]" google-cloud-discoveryengine google-auth-oauthlib

### Restart Kernel

In [ ]:
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

## 1. Load Environment Variables and Initialize Vertex AI


In [2]:
# take environment variables from .env.
load_dotenv()
PROJECT_ID = os.getenv("PROJECT_ID")
LOCATION = os.getenv("LOCATION", "us-central1")
GCS_BUCKET = os.getenv("GCS_BUCKET")
PROJECT_NUMBER = os.getenv("PROJECT_NUMBER")
#REASONING_ENGINE = os.getenv("REASONING_ENGINE")
AS_APP = os.getenv("AS_APP")
AUTH_ID = os.getenv("AUTH_ID")


In [3]:
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=GCS_BUCKET,
)

In [ ]:
# your agent engine resource name
#REASONING_ENGINE

## 2. Generate OAuth 2.0 Authorization URL

In [4]:
CLIENT_ID=os.getenv("CLIENT_ID")

CLIENT_SECRET=os.getenv("CLIENT_SECRET")
#CLIENT_ID

In [5]:
authorization_url=f"https://accounts.google.com/o/oauth2/v2/auth?client_id={CLIENT_ID}&redirect_uri=https%3A%2F%2Fvertexaisearch.cloud.google.com%2Fstatic%2Foauth%2Foauth.html&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbigquery&include_granted_scopes=true&response_type=code&access_type=offline&prompt=consent"

In [6]:
authorization_url

'https://accounts.google.com/o/oauth2/v2/auth?client_id=496235138247-pftjtnfjo1m5uaiknr3c16irobg5g9cl.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fvertexaisearch.cloud.google.com%2Fstatic%2Foauth%2Foauth.html&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbigquery&include_granted_scopes=true&response_type=code&access_type=offline&prompt=consent'

In [7]:
os.environ["OAUTH_AUTH_URI"] = authorization_url


## 3. Get OAuth 2.0 Client ID and Secret
- Get `OAUTH_CLIENT_ID` and `OAUTH_CLIENT_SECRET` from the OAuth client setup in the client_secret.json file
- Ensure that `OAUTH_TOKEN_URI` is what your service requires. Example used here is for Google APIs.

In [ ]:
#client_id = os.getenv("CLIENT_ID")
#client_secret = os.getenv("CLIENT_SECRET")

In [8]:
#os.environ["OAUTH_CLIENT_ID"] = client_id
#os.environ["OAUTH_CLIENT_SECRET"] = client_secret
os.environ["OAUTH_TOKEN_URI"] = "https://oauth2.googleapis.com/token"

In [10]:
%%bash

curl -X POST \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  -H "X-Goog-User-Project: ${PROJECT_NUMBER}" \
https://discoveryengine.googleapis.com/v1alpha/projects/${PROJECT_NUMBER}/locations/global/authorizations?authorizationId=${AUTH_ID} \
  -d '{
  "name": "projects/${PROJECT_NUMBER}/locations/global/authorizations/${AUTH_ID}",
  "serverSideOauth2": {
      "clientId": "'"${CLIENT_ID}"'",
      "clientSecret": "'"${CLIENT_SECRET}"'",
      "authorizationUri": "'"${OAUTH_AUTH_URI}"'",
      "tokenUri": "'"${OAUTH_TOKEN_URI}"'"
    }
  }'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1412    0   698  100   714    704    720 --:--:-- --:--:-- --:--:--  1424-:--:-- --:--:-- --:--:--  1424


{
  "name": "projects/496235138247/locations/global/authorizations/ge-a2a-005",
  "serverSideOauth2": {
    "clientId": "496235138247-pftjtnfjo1m5uaiknr3c16irobg5g9cl.apps.googleusercontent.com",
    "clientSecret": "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",
    "tokenUri": "https://oauth2.googleapis.com/token",
    "authorizationUri": "https://accounts.google.com/o/oauth2/v2/auth?client_id=496235138247-pftjtnfjo1m5uaiknr3c16irobg5g9cl.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fvertexaisearch.cloud.google.com%2Fstatic%2Foauth%2Foauth.html&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbigquery&include_granted_scopes=true&response_type=code&access_type=offline&prompt=consent"
  }
}


In [23]:
import vertexai
from google.genai import types
client = vertexai.Client(
    project=PROJECT_ID,
    location='us-central1',
    http_options=types.HttpOptions(
        api_version="v1beta1", base_url=f"https://us-central1-aiplatform.googleapis.com/"
    ),
)

In [15]:
# Helper for the http based test
from google.auth import default
from google.auth.transport.requests import Request
import httpx

def get_bearer_token():
  try:
    credentials, project = default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
    request = Request()
    credentials.refresh(request)
    return credentials.token
  except Exception as e:
    print(f"Error getting credentials: {e}")
    print("Please ensure you have authenticated with 'gcloud auth application-default login'.")
  return None

In [24]:
remote_a2a_agent_resource_name='projects/496235138247/locations/us-central1/reasoningEngines/1671763449568296960'

In [26]:
config = {"http_options": {"base_url": f"https://{LOCATION}-aiplatform.googleapis.com"}}
remote_a2a_agent = client.agent_engines.get(
    name=remote_a2a_agent_resource_name,
    #config=config,
)

remote_a2a_agent_card = await remote_a2a_agent.handle_authenticated_agent_card()

In [27]:
remote_a2a_agent_card

AgentCard(additional_interfaces=None, capabilities=AgentCapabilities(extensions=None, push_notifications=None, state_transition_history=None, streaming=False), default_input_modes=['text'], default_output_modes=['text'], description='A helpful assistant agent that can answer questions.', documentation_url=None, icon_url=None, name='Hosting Agent - Gemini Enterprise', preferred_transport='HTTP+JSON', protocol_version='0.3.0', provider=None, security=[{'GoogleOAuth': ['https://www.googleapis.com/auth/cloud-platform', 'openid', 'email', 'profile']}], security_schemes={'GoogleOAuth': SecurityScheme(root=OAuth2SecurityScheme(description='OAuth2 for Google Calendar API', flows=OAuthFlows(authorization_code=AuthorizationCodeOAuthFlow(authorization_url='https://accounts.google.com/o/oauth2/auth', refresh_url=None, scopes={'profile': "Access user's basic profile information", 'https://www.googleapis.com/auth/cloud-platform': 'Full access to Google Cloud Platform services', 'openid': 'Authentica

In [9]:
AGENT_URL = os.getenv("AGENT_URL")

In [10]:
agent_card = {
    "url": AGENT_URL,
    "name": "A2A_Test_Agent",
    "preferredTransport": "HTTP+JSON",
    "description": "You're an export of weather and cocktail, answer questions regarding weather and cocktail",
    "capabilities": {"streaming": False},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": [
        {
      "description": "You're an export of weather and cocktail, answer questions regarding weather and cocktail",
      "examples": [
        "Weather in LA, CA",
        "What are ingrediants formargarita?"
      ],
      "id": "question_answer",
      "name": "Q&A Agent",
      "tags": [
        "Question-Answer"
      ]
    }],
    "version": "1.0.0",
    "supportsAuthenticatedExtendedCard": True,
    "protocolVersion": "0.3.0"
}

In [12]:
compact_card = json.dumps(agent_card)

In [29]:
compact_card

'{"additionalInterfaces": null, "capabilities": {"extensions": null, "pushNotifications": null, "stateTransitionHistory": null, "streaming": false}, "defaultInputModes": ["text"], "defaultOutputModes": ["text"], "description": "A helpful assistant agent that can answer questions.", "documentationUrl": null, "iconUrl": null, "name": "Hosting Agent - Gemini Enterprise", "preferredTransport": "HTTP+JSON", "protocolVersion": "0.3.0", "provider": null, "security": [{"GoogleOAuth": ["https://www.googleapis.com/auth/cloud-platform", "openid", "email", "profile"]}], "securitySchemes": {"GoogleOAuth": {"description": "OAuth2 for Google Calendar API", "flows": {"authorizationCode": {"authorizationUrl": "https://accounts.google.com/o/oauth2/auth", "refreshUrl": null, "scopes": {"profile": "Access user\'s basic profile information", "https://www.googleapis.com/auth/cloud-platform": "Full access to Google Cloud Platform services", "openid": "Authenticate user identity", "email": "Access user\'s ema

In [13]:
# @title Helper to register agent on gemini enterprise
import os
import requests
def register_agent_on_gemini_enterprise(
    project_number: str,
    app_id: str,
    agent_card: str,
    agent_name: str,
    display_name: str,
    description: str,
    agent_authorization: str = None,
):
    """Register an Agent Engine to Gemini Enterprise.

    Args:
        project_number: GCP project number
        app_id: Gemini Enterprise application ID
        reasoning_engine_resource_name: Full resource name of deployed Agent Engine
        display_name: Display name for the agent in Gemini Enterprise
        description: Description of the agent

    Returns:
        dict: Response from Discovery Engine API
    """
    raw_location = "global"

    # Map region to Discovery Engine location and base URL
    if raw_location.startswith("us-"):
        location = "us"
        base_url = "https://us-discoveryengine.googleapis.com"
    elif raw_location.startswith("eu-"):
        location = "eu"
        base_url = "https://eu-discoveryengine.googleapis.com"
    else:
        location = "global"
        base_url = "https://discoveryengine.googleapis.com"

    #test-discoveryengine.sandbox.googleapis.com
    # Construct API endpoint
    api_endpoint = (
        f"{base_url}/v1alpha/projects/{project_number}/"
        f"locations/{location}/collections/default_collection/engines/{app_id}/"
        f"assistants/default_assistant/agents"
    )

    #explicitly_escaped_string = compact_card.replace('"', '\\\"')
    #explicitly_escaped_string
    # Prepare request payload
    payload = {
        "name": agent_name,
        "displayName": display_name,
        "description": description,
        "a2aAgentDefinition": {
            "jsonAgentCard": agent_card
        },
    }

    if agent_authorization:
      payload["authorization_config"] = {
          "agent_authorization": agent_authorization
      }

    # Get access token
    bearer_token = get_bearer_token()

    # Prepare headers
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json",
        "X-Goog-User-Project": project_number,
    }

    # Make POST request
    print(f"Registering agent to Gemini Enterprise...")
    print(f"API Endpoint: {api_endpoint}")
    print(f"Display Name: {display_name}")

    response = requests.post(api_endpoint, headers=headers, json=payload)

    if response.status_code == 200:
        print("✓ Agent registered successfully!")
        return response.json()
    else:
        print(f"✗ Registration failed with status code: {response.status_code}")
        print(f"Response: {response.text}")
        response.raise_for_status()

In [16]:
# @title Invoke registration api

ge_project_number = '496235138247' # @param{type:"string"}
ge_app_id = 'ge-app-dw-test1_1764005317392' # @param{type:"string"}
agent_name = 'A2A_ON_GE' # @param{type:"string"}
agent_display_name = 'A2A ON GE' # @param{type:"string"}
agent_description = 'A2A ON GE' # @param{type:"string"}
agent_authorization = '' # @param{type:"string"}
#projects/14326790950/locations/global/authorizations/A2A_ON_GE

register_agent_on_gemini_enterprise(
    project_number=ge_project_number,
    app_id=ge_app_id,
    agent_card=compact_card,
    agent_name=agent_name,
    display_name=agent_display_name,
    description=agent_description,
    agent_authorization=agent_authorization,
)

Registering agent to Gemini Enterprise...
API Endpoint: https://discoveryengine.googleapis.com/v1alpha/projects/496235138247/locations/global/collections/default_collection/engines/ge-app-dw-test1_1764005317392/assistants/default_assistant/agents
Display Name: A2A ON GE
✓ Agent registered successfully!


{'name': 'projects/496235138247/locations/global/collections/default_collection/engines/ge-app-dw-test1_1764005317392/assistants/default_assistant/agents/10440847326271617482',
 'displayName': 'A2A ON GE',
 'description': 'A2A ON GE',
 'createTime': '2025-11-24T21:51:47.991470596Z',
 'state': 'ENABLED',
 'a2aAgentDefinition': {'jsonAgentCard': '{"url": "https://us-central1-aiplatform.googleapis.com/v1beta1/projects/dw-genai-dev/locations/us-central1/reasoningEngines/1671763449568296960/a2a", "name": "A2A_Test_Agent", "preferredTransport": "HTTP+JSON", "description": "You\'re an export of weather and cocktail, answer questions regarding weather and cocktail", "capabilities": {"streaming": false}, "defaultInputModes": ["text/plain"], "defaultOutputModes": ["text/plain"], "skills": [{"description": "You\'re an export of weather and cocktail, answer questions regarding weather and cocktail", "examples": ["Weather in LA, CA", "What are ingrediants formargarita?"], "id": "question_answer", "

## 4. Link Agent to Agentspace

In [11]:
AGENT_URL=os.getenv("AGENT_URL")

In [12]:
import requests
import subprocess

# Get the auth token
auth_token = subprocess.run(
    ["gcloud", "auth", "print-access-token"],
    capture_output=True,
    text=True
).stdout.strip()

# Construct the agent card as a proper JSON object first, then convert to string
agent_card = {
    "url": AGENT_URL,
    "name": "A2A_Test_Agent",
    "preferredTransport": "HTTP+JSON",
    "description": "You're an export of weather and cocktail, answer questions regarding weather and cocktail",
    "capabilities": {"streaming": False},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": [
        {
      "description": "You're an export of weather and cocktail, answer questions regarding weather and cocktail",
      "examples": [
        "Weather in LA, CA",
        "What are ingrediants formargarita?"
      ],
      "id": "question_answer",
      "name": "Q&A Agent",
      "tags": [
        "Question-Answer"
      ]
    }],
    "version": "1.0.0",
    "supportsAuthenticatedExtendedCard": True,
    "protocolVersion": "0.3.0"
}

# Construct the full payload
payload = {
    "name": "A2A",
    "displayName": os.getenv("DISPLAY_NAME", "A2A Agent"),
    "description": os.getenv("DESCRIPTION", "A2A Agent Description"),
    "a2aAgentDefinition": {
        "jsonAgentCard": json.dumps(agent_card)
    },
    "authorization_config": {
        "agent_authorization": f"projects/{PROJECT_NUMBER}/locations/global/authorizations/{AUTH_ID}"
    }
}

# Make the request
url = f"https://discoveryengine.googleapis.com/v1alpha/projects/{PROJECT_NUMBER}/locations/global/collections/default_collection/engines/{AS_APP}/assistants/default_assistant/agents"

headers = {
    "Authorization": f"Bearer {auth_token}",
    "Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)

print(f"Status Code: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

Status Code: 200
Response: {
  "name": "projects/496235138247/locations/global/collections/default_collection/engines/ge-app-dw-test1_1764005317392/assistants/default_assistant/agents/16184770385532636727",
  "displayName": "A2A Agents with Remote MCP",
  "description": "An agent that can answer questions about weather and cocktails.",
  "createTime": "2025-11-24T19:13:13.261449441Z",
  "state": "ENABLED",
  "a2aAgentDefinition": {
    "jsonAgentCard": "{\"url\": \"https://us-central1-aiplatform.googleapis.com/v1beta1/projects/dw-genai-dev/locations/us-central1/reasoningEngines/1671763449568296960/a2a\", \"name\": \"A2A_Test_Agent\", \"preferredTransport\": \"HTTP+JSON\", \"description\": \"You're an export of weather and cocktail, answer questions regarding weather and cocktail\", \"capabilities\": {\"streaming\": false}, \"defaultInputModes\": [\"text/plain\"], \"defaultOutputModes\": [\"text/plain\"], \"skills\": [{\"description\": \"You're an export of weather and cocktail, answer 

## 5. Try your agent in Agentspace UI
1. Ensure that `Vertex AI API` and `Discovery Engine API` are enabled in your project and that Discovery Engine service account has `Vertex AI User` permission. If you cannot find it in IAM, check the `Include Google-provided role grants`.
2. Open up the Agentspace app in Google Cloud console
3. Select your Agentspace app
3. Click `Copy URL`
4. Open the URL in a new tab
5. From left menu, select your agent from `Agents` deployed

## 6. List agents (Optional)

In [17]:
%%bash
# export PROJECT_NUMBER=PROJECT_NUMBER
# export AS_APP=AGENTSAPCE_APP_ID
curl -X GET -H "Authorization: Bearer $(gcloud auth print-access-token)" \
-H "Content-Type: application/json" \
-H "X-Goog-User-Project: ${PROJECT_NUMBER}" \
"https://discoveryengine.googleapis.com/v1alpha/projects/${PROJECT_NUMBER}/locations/global/collections/default_collection/engines/${AS_APP}/assistants/default_assistant/agents"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

{
  "agents": [
    {
      "name": "projects/496235138247/locations/global/collections/default_collection/engines/ge-app-dw-test1_1764005317392/assistants/default_assistant/agents/10440847326271617482",
      "displayName": "A2A ON GE",
      "description": "A2A ON GE",
      "createTime": "2025-11-24T21:51:48.345319Z",
      "updateTime": "2025-11-24T21:51:48.345319Z",
      "state": "ENABLED",
      "a2aAgentDefinition": {
        "jsonAgentCard": "{\"url\": \"https://us-central1-aiplatform.googleapis.com/v1beta1/projects/dw-genai-dev/locations/us-central1/reasoningEngines/1671763449568296960/a2a\", \"name\": \"A2A_Test_Agent\", \"preferredTransport\": \"HTTP+JSON\", \"description\": \"You're an export of weather and cocktail, answer questions regarding weather and cocktail\", \"capabilities\": {\"streaming\": false}, \"defaultInputModes\": [\"text/plain\"], \"defaultOutputModes\": [\"text/plain\"], \"skills\": [{\"description\": \"You're an export of weather and cocktail, answer qu

100  4733    0  4733    0     0   9154      0 --:--:-- --:--:-- --:--:--  9154


rise users by combining advanced AI with a unique tournament-style competition framework to generate and rank ideas",
      "createTime": "2025-11-24T17:28:52.300492Z",
      "updateTime": "2025-11-24T17:28:52.300492Z",
      "managedAgentDefinition": {
        "toolSettings": {
          "toolDescription": "This agent is a specialized agent that helps with innovation and problem-solving for enterprise users by combining advanced AI with a unique tournament-style competition framework to generate and rank ideas"
        }
      },
      "state": "ENABLED",
      "sharingConfig": {
        "scope": "ALL_USERS"
      }
    }
  ]
}


## 7. Delete an agent (when you don't need it anymore )

In [18]:
os.environ["AGENT_RESOURCE_NAME"] = "projects/496235138247/locations/global/collections/default_collection/engines/ge-app-dw-test1_1764005317392/assistants/default_assistant/agents/10440847326271617482"

In [19]:
%%bash
# export PROJECT_NUMBER=PROJECT_NUMBER
# export AGENT_RESOURCE_NAME=AGENT_RESOURCE_NAME

curl -X DELETE \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  -H "X-Goog-User-Project: ${PROJECT_NUMBER}" \
https://discoveryengine.googleapis.com/v1alpha/${AGENT_RESOURCE_NAME}

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   241    0   241    0     0    293      0 --:--:-- --:--:-- --:--:--   293 0 --:--:-- --:--:-- --:--:--   293


{
  "name": "projects/496235138247/locations/global/collections/default_collection/engines/ge-app-dw-test1_1764005317392/assistants/default_assistant/agents/10440847326271617482/operations/delete-agent-5155658520739663844",
  "done": true
}
